# The vLLM foundations vLLM-Omni inherits

## Learning goals
- Understand prefill, decode, KV cache, and continuous batching
- Explain PagedAttention conceptually
- Know which optimizations apply to AR stages

> **Mac track:** executable cells use lightweight simulations. Boxes labelled **Linux GPU lab** show how the same concept maps to the official runtime.

## Prefill and decode

- **Prefill:** process all prompt tokens and populate attention state.
- **Decode:** generate one or a few new tokens iteratively.
- **KV cache:** retain attention keys and values so old tokens are not recomputed.
- **Continuous batching:** admit and retire requests dynamically instead of waiting for a fixed batch.

In [ ]:
requests={"A":5,"B":2,"C":4}
step=0
while requests:
    step+=1
    active=list(requests)
    print("decode step",step,"batched requests",active)
    for r in active:
        requests[r]-=1
        if requests[r]==0: del requests[r]

## PagedAttention intuition

Treat KV memory as fixed-size blocks instead of requiring one contiguous allocation per sequence. Logical token positions map to physical blocks, reducing fragmentation and enabling sharing.

```text
Request A logical blocks: [A0, A1, A2]
Physical memory:           [B0, A2, free, A0, B1, A1]
```

vLLM-Omni reuses these mechanisms inside autoregressive stages; it does not apply a language KV cache to every DiT or vocoder operation.

In [ ]:
block_size=16
lengths=[10,17,40,65]
for n in lengths:
    blocks=(n+block_size-1)//block_size
    print(n,"tokens ->",blocks,"blocks; reserved",blocks*block_size)

## Optional real Mac lab

Install **vLLM-Metal** following its versioned official documentation, serve a supported small text model, and call its OpenAI-compatible endpoint. This demonstrates the AR foundation, not full vLLM-Omni.